# H4 · `scripts/sweep.py`

## What this file is for

`run_config` -- one configuration, rated across a list of jurisdictions, offline or against ISO.
Everything above it in this set (`H2 qa.py`, the layered programme) is a different way of choosing
*which* configurations to hand this one function; this is the only thing that actually calls the
kernel and, when asked, ISO.

**Three outcomes, not two.** `NOT APPLICABLE` -- a jurisdiction that cannot express the
configuration at all -- is a distinct result from disagreeing with ISO. Counting it as a failure
would report twenty false negatives for a risk ISO never permitted, which buries the number that
matters: do we agree where the question is legal?

**Engine-only by default.** All 51 rate offline in about ninety seconds; `--live` adds one real ISO
call per jurisdiction. Agreement itself is not defined here -- `phase2_compare.compare_payload`
owns that, so two definitions of *agree* can never drift apart.

**Depends on:** [`H1 variants.py`](01-variants.ipynb) to build a payload,
[`H5 runstore.py`](05-runstore.ipynb) to record what happened.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import sweep

for name, obj in vars(sweep).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != sweep.__name__:
        continue
    if inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        v = repr(obj)
        print(f"{name} = {v if len(v) < 90 else v[:87] + '...'}")

## The smallest thing that works

`run_config({}, jurisdictions)` rates the unvaried base risk everywhere named, offline, and hands
back a row per state plus a summary.

In [ ]:
out = sweep.run_config({"occurrence_limit": "2,000,000 CSL"}, ["TX", "CA", "AK"], compare=False)
for r in out["rows"]:
    print(f"  {r['juris']:4s} {r['status']:8s} premium {r.get('ours')}")
print()
print("summary:", {k: out["summary"][k] for k in
                   ("total", "rated", "not_applicable", "seconds")})

## The interesting case

### `resolve` turns a position into a figure, per state, and records what it got

Naming a fixed aggregate like *10,000,000* and sweeping 51 states does not work -- the legal set is
keyed on the occurrence limit and differs by state, so a figure legal in Texas can be undeliverable
in Alaska. `resolve(config, declared)` runs **per jurisdiction, before the payload is built**, and
what it actually chose is written onto that state's row -- because a run that stores the request and
not the answer cannot be read back a week later, and *"the highest aggregate"* is a different number
in every state.

In [ ]:
def highest_aggregate(cfg, d):
    occ = cfg.get("occurrence_limit", "1,000,000 CSL")
    legal = d.aggregates_for(occ)
    out = dict(cfg)
    if legal:
        out["general_aggregate"] = legal[-1]           # the position: @highest
    return out

out = sweep.run_config({"occurrence_limit": "1,000,000 CSL"}, ["TX", "CA", "AK"],
                       compare=False, resolve=highest_aggregate)
for r in out["rows"]:
    print(f"  {r['juris']:4s} resolved={r.get('resolved')}  premium={r.get('ours')}")

### A stopped run is not a failed run

`stop_check()` runs before each jurisdiction; a truthy return ends the run there. What it never
reached is named, not silently dropped -- so a person reviewing the run later can tell *"stopped
early"* from *"actually covers only three states"*.

In [ ]:
seen = {"n": 0}
def stop_after_two():
    seen["n"] += 1
    return seen["n"] > 2

out = sweep.run_config({}, ["TX", "CA", "AK", "NY"], compare=False,
                       stop_check=stop_after_two)
print("stopped_early:", out["summary"]["stopped_early"])
print("rated        :", [r["juris"] for r in out["rows"]])
print("not_reached  :", out["summary"]["not_reached"])

## What it refuses

A jurisdiction that cannot express the configuration at all -- Alaska declares one prem/ops
territory, so a second location is not a disagreement to report, it is `NOT APPLICABLE`.

In [ ]:
out = sweep.run_config({"locations": 2}, ["AK"], compare=False)
r = out["rows"][0]
print(r["status"], "--", r["detail"])

## Try it yourself

1. Set `probe=False` on a call that would otherwise leave the premium unchanged -- what disappears
   from the row, and why is that expensive to skip on a large sweep?
2. `python scripts/sweep.py --controls` from a terminal lists every control's live options in one
   jurisdiction -- the same declaration [`H1`](01-variants.ipynb) reads, seen from the CLI.
3. `baselines()` caches the unvaried premium per jurisdiction to disk. Delete
   `scripts/erc/out/baselines.json` and re-run a cell above -- what changes about the timing, and
   why is the file safe to delete at any time?

In [ ]:
# your turn